# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, accessible at the following URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant pandas matplotlib

## 1. Data Loading
Load dataset metadata and inspect its core information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Set the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Dataset description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}\n")

## 2. Data Overview
Let's examine the available record sets (tables), fields, and their unique Croissant `@id` values.

`mlcroissant` enables listing all record sets and schema elements available in the dataset—these will act as entry points for loading records.

In [ ]:
# List available record sets by @id and name
if hasattr(dataset, "record_sets") and dataset.record_sets:
    print("Available record sets:")
    for rs in dataset.record_sets:
        print(f"  - @id: {rs['@id']}")
        print(f"    name: {rs.get('name')}")
        # Optionally print field @ids and names
        if 'field' in rs and rs['field']:
            print("    Fields:")
            for field in rs['field']:
                field_obj = field if isinstance(field, dict) else dataset.field_by_id(field)
                print(f"      > @id: {field_obj['@id']}, name: {field_obj.get('name')}")
        print()
else:
    # Fallback, retrieve via dataset metadata
    print("No record sets found in Croissant metadata.")

## 3. Data Extraction
Load records from the available record sets using their `@id`s. 

We'll identify all record set `@id`s available, then extract their data using `dataset.records(record_set=...)`. Results for each table are loaded into a pandas DataFrame.

For demonstration, we extract from each record set and display columns for the first table.

In [ ]:
# Get all record set @ids
record_sets = []
if hasattr(dataset, "record_sets") and dataset.record_sets:
    record_sets = [rs['@id'] for rs in dataset.record_sets]
else:
    print("No record sets defined in the Croissant schema.")

dataframes = {}
for rs_id in record_sets:
    # Load all records from this record set
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set @id: {rs_id}")
    except Exception as e:
        print(f"Could not load records from record set @id: {rs_id} -- {str(e)}")

# If there is at least one extracted dataframe, print columns of the first one
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns for first record set (@id: {first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No data available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Let's perform simple exploratory steps on numeric fields. We'll pick the first record set and select a numeric field by inspecting the dataframe. Then, we'll filter, normalize, and optionally group by a categorical column if present.

All Croissant entity references (record sets and fields) are by their `@id`.


In [ ]:
# --- EDA on first available record set ---
import numpy as np

# Choose the first loaded record set
if dataframes:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]

    # Try to pick a numeric field (column with dtype int or float)
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: '{numeric_field}' from record set '@id': {rs_id}")

        # Filter: keep values > threshold
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (total: {len(filtered_df)})")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_field = f"{numeric_field}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (z-score) for filtered records:")
        display(filtered_df[[numeric_field, norm_field]].head())

        # Try grouping by a categorical column, if available
        cat_fields = df.select_dtypes(include=["object", "category"]).columns
        group_field = None
        for c in cat_fields:
            if df[c].nunique() < 20 and c != numeric_field:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found in the first record set.")
else:
    print("No data loaded; cannot perform EDA.")

## 5. Visualization
Create basic visualizations of a numeric field distribution or the relationship between key attributes.

All axes and legends should specify field `@id` when possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals() and not filtered_df.empty:
    # Histogram of the normalized numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[norm_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} (normalized) in record set '@id': {rs_id}")
    plt.xlabel(f"{numeric_field} (z-score normalization)")
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field exists, plot group means
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,5))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field} in '@id': {rs_id}")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR² dataset using `mlcroissant`:
- Dataset metadata and structure were inspected.
- Record sets, fields, and their Croissant `@id`s were used for querying.
- Data was loaded into pandas DataFrames, explored, filtered, and normalized by field `@id`.
- Simple visualizations highlighted the distribution and grouped statistics of selected numeric variables.

For further exploration, review the dataset's full Croissant schema for detailed field documentation, and extend the notebook with domain-specific analyses as needed.